# 📨 Notebook 1 — The Problem: Writes Vanish When a Replica Is Down

Before we learn what **hinted handoff** is, we need to *feel* the problem it solves.

## 🏘️ The analogy: three identical mailboxes

Imagine you live in a neighborhood with three *identical* mailboxes (`r1`, `r2`, `r3`) that are supposed to hold a copy of every letter.
A mail carrier (the **coordinator**) drops a copy of each letter into every mailbox. That way, if one mailbox is destroyed in a fire, the other two still have your mail. This is **replication**.

But what happens when one mailbox is **temporarily out of service** (e.g., the owner locked it)?
In a naive system, the carrier just *skips* it. Every letter that arrives during the outage is **lost forever** for that mailbox. The two other mailboxes have it, but the broken one will be out of sync.

That is exactly what happens in a distributed database without hinted handoff. Let's see it in code.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/hinted-handoff
uv sync
```

Then in VS Code, click the kernel picker at the top-right of this notebook and choose **`.venv`**.
If it doesn't show up, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🎯 What you'll learn in this notebook

1. How a coordinator forwards writes to replicas.
2. Why *forward-and-forget* silently loses data when a replica is down.
3. How this creates stale reads until a (slow, expensive) background repair runs.

This is the **bad** baseline. Notebook 2 introduces the fix.

## 🟥 BAD: forward-and-forget

We model three replicas. The coordinator tries to write to all of them. If a replica is down, the write is simply dropped. No one remembers that this write ever happened for the downed replica.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict

@dataclass
class Replica:
    """A single storage node. Holds a tiny in-memory key/value store."""
    name: str
    up: bool = True                          # is the node reachable right now?
    data: Dict[str, str] = field(default_factory=dict)

    def write(self, key: str, value: str) -> bool:
        if not self.up:
            # Network error / crashed process — the write never lands.
            print(f'  ⚠ {self.name} is DOWN — write ({key}={value}) dropped')
            return False
        self.data[key] = value
        print(f'  ✅ {self.name} stored ({key}={value})')
        return True

replicas = [Replica('r1'), Replica('r2'), Replica('r3')]

def coordinator_write(key: str, value: str) -> None:
    """Naive coordinator: fan-out to every replica, ignore failures."""
    print(f'coordinator: write {key}={value}')
    for r in replicas:
        r.write(key, value)


Now simulate an outage: `r3` dies, five writes arrive, then `r3` comes back.

In [ ]:
replicas[2].up = False            # r3 goes offline
print('--- r3 is DOWN; 5 writes arrive ---')
for i in range(5):
    coordinator_write(f'k{i}', f'v{i}')

print('\n--- r3 comes back online ---')
replicas[2].up = True

print('\nFinal state of each replica:')
for r in replicas:
    print(f'  {r.name}: {r.data}')


### 🧐 What do you see?

- `r1` and `r2` have all five keys (`k0…k4`).
- `r3` has **nothing** — it lost every write that happened during its outage.

Right now, if a client reads `k2` from `r3`, it will get a **miss** (or, worse, an old value from a previous generation).
This is called a **stale replica**, and until it is repaired, the cluster is *inconsistent*.

## 🔴 Why this hurts in the real world

In [ ]:
# Simulate a client that happens to read from r3 after recovery.
def client_read(key, replica):
    v = replica.data.get(key)
    status = 'HIT ' + repr(v) if v is not None else '❌ MISS'
    print(f'  client reads {key} from {replica.name} → {status}')

for k in ['k0', 'k2', 'k4']:
    client_read(k, replicas[2])


Every miss is a user-visible bug: a missing order, a forgotten signup, a shopping cart that lost an item.

### ⏳ "But won't anti-entropy repair eventually fix this?"

Yes — eventually. Real systems do run a **Merkle-tree / anti-entropy scan** in the background to compare replicas and sync them. 
But that scan is:

- **Expensive**: it reads every key on every replica.
- **Slow**: typically runs on a schedule (hours / days).
- **Reactive**: clients see stale data until the next scan completes.

We want a **fast, cheap, targeted** mechanism to catch up just the writes a recovering node missed. That's **hinted handoff** — coming up in Notebook 2.

👉 Continue to [`02_hinted_handoff.ipynb`](./02_hinted_handoff.ipynb).